## Original SCB agent prompt provided in the paper
### Before the modular impl. agents are implemented, use this 
Specify additionally that follow the design in `current_design.json`

In [5]:
from prompts.scb import get_original_scb_prompt

# Pass the current checkpoint number to be implemented 
print(get_original_scb_prompt(1))


Implement a program that 100% solves the specification.
That is all you need to do.

Use a virtual environment and ensure that a 'requirements.txt' is present with any dependencies
you need to solve the problem.

You are working on the following issue:
Issue path: checkpoint_1.md
Issue implementation path: checkpoint_1/




## Reader agent (Unnecessary because we have metrics?)
Its purpose is to intiialise the `current_design.json` for the loop, and does not output anything. 

In [ ]:
from prompts.reader import get_reader_prompt

print(get_reader_prompt(3))


You are a senior software engineer that analyses modules in a software project.

Your job is to understand the following directory: 
Project root: agent_workspace
Directory: checkpoint_2/
Dependency graph: checkpoint_2_graph.json

Identify the existing modules in the codebase, using the dependency graph as reference. For each module, identify its responsibility. Do not propose new modules.

Write a JSON object to `current_design.json`, that describes each module using the following schema. If there are no modules, write an empty JSON array.
{
  "type": "array",
  "items": {
    "type": "object",
    "properties": {
      "module_name": {
        "description": "The name of the module.",
        "type": "string"
      },
      "module_path": {
        "description": "The directory path of the module.",
        "type": "string"
      },
      "responsibility": {
        "description": "A sentence describing the single responsibility of this module.",
        "type": "string"
      }
   

## Analyzer agent

In [4]:
from prompts.analyzer import get_analyzer_prompt

# The actual DPy results should be filtered for specific things we concern only. 
# For this TEST example, a path is provided instead. DO NOT do this in your final submission 
print(get_analyzer_prompt(second_iteration=False))


You are a senior software code quality analyst.

Your job is to evaluate the following existing modules: 
Project root: agent_workspace
Design: `current_design.json`
Dependency graph: `current_deps_graph.json`

Your evaluation is based on the following criteria to achieve best code quality and maintainability: 
- Modules should be able to independently evolve, with low coupling and high cohesion 
- The system should be easy to test by not having overly complex functions with lots of control paths that could be simplified
- The maintenance effort when introducing new features should be as low as possible by avoiding duplication and ensuring single responsibility

For each module, reason about the following: 
- Is the module likely to change when new features are added with later checkpoints? 
- If a new feature is added, how much effort would it take to ensure consistent behaviour, such as having to modify multiple unrelated files or extend complex logic? 
- Does the nature and complex

## Decomposer Agent

In [6]:
from prompts.decomposer import get_decomposer_prompt

# Extract only the improvements part of the analyzer output 
print(get_decomposer_prompt(1 , second_iteration=True))


You are a senior software engineer that specialises in modular software design.

You are working on the following issue:
Project root: agent_workspace
Issue path: checkpoint_1.md


You are also given the current modules design in the file `current_design.json`, prioritise reusing existing modules instead of creating a new module where possible.
The current modular design is visualised by the dependency graph in `current_deps_graph.json`.
In addition, you are given a list of improvement suggestions on the current design in `current_analyzer_result.json`, please factor them in your design together with the new modules if the improvement would improve code quality in the long run, or reject it if the improvement is not applicable.

Propose a modular design that achieves the goal specified in the issue when integrated together. 
You should follow best code practices, including: 
- A module should only expose the minimum amount of knowledge in its public interface 
- Each module should onl

## Analyzer second time

In [4]:
from prompts.analyzer import get_analyzer_prompt

print(get_analyzer_prompt(second_iteration=True))


You are a senior software code quality analyst. 

Your job is to analyse the following modular design including kept, changed or new modules: 
Project root: agent_workspace
Design: `current_design.json`
Dependency graph: `current_deps_graph.json`

A list of flagged code smells have been given in `current_metrics`. 
You are also given a list of previously suggested improvements that are rejected in `current_rejected_improvements.json`.

Interpret the metrics using the following guidelines: 
- LCOM: Only flag if the methods have different responsibilities. It is acceptable if the methods are just sequential stages of the same functionality. 
- Cyclomatic complexity, WMC: Only flag if the complexity is caused by unrelated concerns, lead to low testability, or high maintenance effort when adding new features. It is acceptable if the complex problem logic justifies it. 
- Magic number: Only flag if the meaning of the number is not immediately obvious to the developer. 
- Number of public m

In [7]:
ANALYZER_OUTPUT_2 = {
    "result": "pass",
    "improvements": [
      {
        "module_name": "pipeline.ast_nodes",
        "smell": "Module conflates language-level AST node types with application-level caching configuration structures, reducing cohesion and coupling the language frontend to the caching subsystem.",
        "improvement_instruction": "Extract TtlConfig, CacheKeyConfig, CacheConfig, and GlobalCacheConfig into a dedicated pipeline.cache_config module. pipeline.ast_nodes should retain only constructs that represent parsed language elements (TaskDef, ParamDef, and token-adjacent types). Update imports in pipeline.cache_key, pipeline.cache_manager, pipeline.executor, and pipeline.main accordingly."
      },
      {
        "module_name": "pipeline.expr_parser",
        "smell": "parse_block returns List[Any], erasing all type information at the parser/evaluator boundary despite typed AST node dataclasses existing in pipeline.ast_nodes.",
        "improvement_instruction": "Define a StmtNode union type or a common base dataclass in pipeline.ast_nodes that covers all statement node variants (IfStmt, ForStmt, WhileStmt, AssignStmt, ReturnStmt, etc.). Change ExprParser.parse_block to return List[StmtNode] so that static type checking is preserved across the parse/evaluate boundary."
      },
      {
        "module_name": "pipeline.evaluator",
        "smell": "eval_block accepts raw List[Token] and internally invokes ExprParser, conflating token parsing with expression evaluation and violating the established lex-parse-evaluate layering.",
        "improvement_instruction": "Remove the token-to-AST parsing step from eval_block. Change its signature to accept List[StmtNode] (a pre-parsed AST). Callers such as Executor should invoke ExprParser.parse_block explicitly before calling eval_block, keeping the two phases separately testable and aligned with the pipeline.expr_parser/pipeline.evaluator module boundary."
      },
      {
        "module_name": "pipeline.cache_manager",
        "smell": "store() accepts both the precomputed cache_key and the raw inputs (task_def, params, workspace) from which the key was derived, producing a redundant and inconsistent method signature.",
        "improvement_instruction": "Simplify store() to accept only task_def (for cache location resolution), cache_key, and job_result. Remove the redundant params and workspace parameters; since cache_key is already computed by a prior check() call, only the cache directory (derivable from task_def.cache.location) is needed to persist the entry."
      },
      {
        "module_name": "pipeline.cache_store",
        "smell": "The exists() method is fully subsumed by load() returning None, unnecessarily widening the public interface and enabling TOCTOU access patterns.",
        "improvement_instruction": "Remove the exists() method from cache_store's public interface. Update all callers in pipeline.cache_manager to use load() and branch on the None return value, eliminating the separate existence check."
      }
    ]
  }

# Modular coder agent

In [ ]:
from prompts.modular_coder import get_modular_coder_prompt

print(
    get_modular_coder_prompt(1,['pipeline/requires_executor.py', 'pipeline/success_evaluator.py'])
)


You are working on the following issue:
Issue path: checkpoint_1.md
Implement your solution in: checkpoint_1/


Use a virtual environment and ensure that a 'requirements.txt' is present with any dependencies
you need to solve the problem.

Ensure your code a good quality, below are a list of practices you should follow but not limited to them. 
- Avoids functions that are too complex with too much nested if/else statements.
- Avoid the use of magic numbers when their meanings are not obvious.
- Do not access the private elements of another class.


Your job is to implement parts of the design specified in `current_design.json`, and you should import existing modules. 
You should ONLY implement the following 2 modules: 
- pipeline/requires_executor.py
- pipeline/success_evaluator.py


